# Imports

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from master_thesis.config import DATA_PROCESSED

In [2]:
import pandas as pd
import numpy as np

from master_thesis.data_utils import load_processed, save_processed

In [3]:
df_final_no_spend = load_processed(
    "contract_with_gold.csv",
    low_memory=False,
    dtype={
        "terminated": "str",
        "supplier_number": "str",
    },
)

print("Input shape:", df_final_no_spend.shape)
display(df_final_no_spend.head())

Input shape: (9201, 147)


,contract_id,contract_number,contract_name,contract_status,contract_owner_cost_centre,terminated,term_type,start_date,expiration_date,supplier_id,...,LPI_Score,Customs,Infrastructure,International_Shipments,Logistics_Competence,Tracking_Tracing,Timeliness,PPI_Value,gold_y,gold_department
0,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,84.4,NaN,NaN
1,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,83.9,NaN,NaN
2,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,79.9,NaN,NaN
3,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,100.0,NaN,NaN
4,9675,9675,Bioreliance_Master_2018_MSA,published,7756.0,False,fixed,2018-05-21 00:00:00+00:00,2027-07-30 00:00:00+00:00,11544,...,3.991859,3.918058,3.95873,3.530159,4.007843,4.176078,4.407843,147.0,NaN,NaN


# Feature engineering

In [4]:
df_features = df_final_no_spend.copy()

In [5]:
print(df_features.shape)
print(df_features.columns.tolist())

(9201, 147)
['contract_id', 'contract_number', 'contract_name', 'contract_status', 'contract_owner_cost_centre', 'terminated', 'term_type', 'start_date', 'expiration_date', 'supplier_id', 'supplier_number', 'supplier_display_name', 'payment_terms', 'incoterms', 'contract_currency_code', 'contract_value', 'contract_value_dkk', 'contract_type', 'nn_contract_type', 'contract_commodity', 'team', 'unit', 'company_country', 'start_year', 'expiry_year', 'open_ended_contract', 'end_year', 'start_year_capped', 'observation_year', 'years_to_expiry', 'contract_age_years', 'expiry_pressure_bucket', 'department_from_cost_center', 'department', 'moodys_bvd_id', 'fin_moodys_company_name', 'fin_closing_year', 'fin_closing_date', 'fin_created_at_utc', 'fin_Status', 'fin_Implied_rating', 'fin_risk_level', 'fin_Financial_level', 'fin_output_text', 'fin_Implied_rating - original', 'fin_Number_of_months', 'fin_Net_Salesth_DKK', 'fin_Operating_revenue_Turnover_th_DKK', 'fin_Gross_profit_th_DKK', 'fin_EBIT_t

In [6]:
df_contracts_per_department = (
    df_features
    .groupby("department", dropna=False)["contract_id"]
    .nunique()
    .reset_index(name="unique_contract_count")
    .sort_values("unique_contract_count", ascending=False)
)

print(df_contracts_per_department)

                                 department  unique_contract_count
3                         Devices & Needles                    476
4                  Drug Product Outsourcing                    300
5                Drug Substance Outsourcing                    298
9                        Packaging Material                    289
11                   Raw Materials & Energy                    281
10  Quality, Production Services & Supplies                    215
1             Bioprocessing & Raw Materials                    113
8                                 Logistics                     94
2              Bioprocessing and Excipients                     59
0          Alliance, Acquisitions & PPM CoE                     35
14                                      NaN                     22
12                Strategic Sourcing US Hub                      9
6    Global Strategic Outsourcing & Devices                      8
7            HI Warehouse Expansion Program                   

## Financial Features

Helper functions

In [7]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    """Safely divide two Series and return NaN where denominator is 0 or missing."""
    denominator = denominator.replace(0, np.nan)
    return numerator / denominator


def make_missing_flag(df: pd.DataFrame, col: str, flag_name: str | None = None) -> pd.DataFrame:
    """Create a missingness flag for a column."""
    if flag_name is None:
        flag_name = f"{col}_missing_flag"
    df[flag_name] = df[col].isna().astype(int)
    return df


Standardize columns

In [8]:
financial_column_map = {
    "fin_Implied_rating": "fin_implied_rating",
    "fin_risk_level": "fin_risk_level",
    "fin_Financial_level": "fin_financial_level",
    "fin_financial_risk_score": "fin_financial_risk_score",
    "fin_Current_ratio": "fin_current_ratio",
    "fin_Gearing": "fin_gearing",
    "fin_Long_term_Gearing": "fin_long_term_gearing",
    "fin_Short_term_Gearing": "fin_short_term_gearing",
    "fin_Debt_Asset_ratio": "fin_debt_asset_ratio",
    "fin_Interest_coverage_ratio": "fin_interest_coverage_ratio",
    "fin_Solvency_ratio_Asset_based": "fin_solvency_ratio_asset_based",
    "fin_Long_term_liabilities_Equity_ratio": "fin_long_term_liabilities_equity_ratio",
    "fin_Short_term_liabilities_Equity_ratio": "fin_short_term_liabilities_equity_ratio",
    "fin_Profit_margin": "fin_profit_margin",
    "fin_EBIT_margin": "fin_ebit_margin",
    "fin_ROE_using_Net_income": "fin_roe_net_income",
    "fin_ROA_using_Net_income": "fin_roa_net_income",
}

for old_col, new_col in financial_column_map.items():
    if old_col in df_features.columns and new_col not in df_features.columns:
        df_features = df_features.rename(columns={old_col: new_col})

Insure numerical values 

In [9]:
numeric_fin_cols = [
    "fin_financial_risk_score",
    "fin_current_ratio",
    "fin_gearing",
    "fin_long_term_gearing",
    "fin_short_term_gearing",
    "fin_debt_asset_ratio",
    "fin_interest_coverage_ratio",
    "fin_solvency_ratio_asset_based",
    "fin_long_term_liabilities_equity_ratio",
    "fin_short_term_liabilities_equity_ratio",
    "fin_profit_margin",
    "fin_ebit_margin",
    "fin_roe_net_income",
    "fin_roa_net_income",
]

for col in numeric_fin_cols:
    if col in df_features.columns:
        df_features[col] = pd.to_numeric(df_features[col], errors="coerce")

Missing flags

In [10]:
missing_flag_cols = [
    "fin_implied_rating",
    "fin_risk_level",
    "fin_financial_level",
    "fin_financial_risk_score",
    "fin_current_ratio",
    "fin_gearing",
    "fin_short_term_gearing",
    "fin_long_term_gearing",
    "fin_interest_coverage_ratio",
    "fin_profit_margin",
    "fin_ebit_margin",
]

for col in missing_flag_cols:
    if col in df_features.columns:
        df_features = make_missing_flag(df_features, col, f"{col}_missing_flag")

### Rating / summarized risk features

The first group of financial features captures **high-level external or summarized risk assessments**. These variables are useful because they compress complex financial information into interpretable risk categories that can be translated into weak labeling rules.

In this project, the most important variables in this category are:

- `fin_implied_rating`
- `fin_risk_level`
- `fin_financial_level`
- `fin_financial_risk_score`

In [11]:
if "fin_implied_rating" in df_features.columns:
    rating_order = {
        "Aaa": 1,
        "Aa": 2,
        "A": 3,
        "Baa": 4,
        "Ba": 5,
        "B": 6,
        "Caa": 7,
        "Ca": 8,
        "C": 9,
    }
    df_features["fin_implied_rating_ordinal"] = df_features["fin_implied_rating"].map(rating_order)

    df_features["fin_flag_weak_implied_rating"] = (
        df_features["fin_implied_rating"].isin(["Ba", "B", "Caa", "Ca", "C"])
    ).astype(int)

    df_features["fin_flag_moderate_or_worse_rating"] = (
        df_features["fin_implied_rating"].isin(["Baa", "Ba", "B", "Caa", "Ca", "C"])
    ).astype(int)

if "fin_risk_level" in df_features.columns:
    df_features["fin_flag_risk_take_caution_or_worse"] = (
        df_features["fin_risk_level"].isin(["Take caution", "Do not source"])
    ).astype(int)

    df_features["fin_flag_risk_do_not_source"] = (
        df_features["fin_risk_level"].eq("Do not source")
    ).astype(int)

if "fin_financial_level" in df_features.columns:
    df_features["fin_flag_high_financial_risk_level"] = (
        df_features["fin_financial_level"].isin(["High financial risk", "Very high financial risk"])
    ).astype(int)

if "fin_financial_risk_score" in df_features.columns:
    df_features["fin_flag_financial_risk_score_high"] = (
        df_features["fin_financial_risk_score"] >= 3
    ).astype(int)

    df_features["fin_flag_financial_risk_score_very_high"] = (
        df_features["fin_financial_risk_score"] >= 4
    ).astype(int)

### Liquidity stress features

Liquidity stress reflects whether a supplier can meet its short-term obligations. A standard and widely used liquidity measure is the **current ratio**, defined as:


$\text{Current Ratio} = \frac{\text{Current Assets}}{\text{Current Liabilities}}$




The current ratio measures a firm's ability to cover short-term obligations with short-term assets. Lower values indicate weaker liquidity and potentially higher short-term financial pressure.  [oai_citation:1‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/accounting/current-ratio-formula/?utm_source=chatgpt.com)

In the feature engineering step, the raw current ratio is kept, but it is also converted into interpretable stress flags such as:

- `fin_flag_liquidity_stress`
- `fin_flag_severe_liquidity_stress`
- `fin_flag_strong_liquidity`

In [12]:
if "fin_current_ratio" in df_features.columns:
    df_features["fin_flag_liquidity_stress"] = (
        df_features["fin_current_ratio"] < 1
    ).astype(int)

    df_features["fin_flag_severe_liquidity_stress"] = (
        df_features["fin_current_ratio"] < 0.75
    ).astype(int)

    df_features["fin_flag_strong_liquidity"] = (
        df_features["fin_current_ratio"] >= 2
    ).astype(int)

### Leverage / liability stress features

Leverage features aim to capture whether a supplier is under pressure from debt and liabilities relative to equity or assets. This is relevant because highly leveraged firms may be more exposed to refinancing pressure, reduced flexibility, or short-term commercial sensitivity.

A common leverage ratio is the debt-to-equity ratio (also described as gearing), typically expressed as:


$\text{Debt-to-Equity Ratio} = \frac{\text{Total Debt}}{\text{Shareholders' Equity}}$


A related solvency-oriented ratio is the debt-to-asset ratio:


$\text{Debt-to-Asset Ratio} = \frac{\text{Total Debt}}{\text{Total Assets}}$


These ratios are standard measures of leverage and financial burden. Higher values indicate greater reliance on debt financing and potentially greater financial pressure.  [oai_citation:2‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/commercial-lending/debt-to-equity-ratio-formula/?utm_source=chatgpt.com)

In this project, leverage is represented through features such as:

- `fin_gearing`
- `fin_long_term_gearing`
- `fin_short_term_gearing`
- `fin_debt_asset_ratio`
- `fin_long_term_liabilities_equity_ratio`
- `fin_short_term_liabilities_equity_ratio`

These are engineered into stress indicators such as:

- high gearing
- high short-term gearing
- high debt-to-asset burden
- high liabilities-to-equity burden

In [13]:
if "fin_gearing" in df_features.columns:
    df_features["fin_flag_gearing_high"] = (
        df_features["fin_gearing"] > 100
    ).astype(int)

if "fin_long_term_gearing" in df_features.columns:
    df_features["fin_flag_long_term_gearing_high"] = (
        df_features["fin_long_term_gearing"] > 100
    ).astype(int)

if "fin_short_term_gearing" in df_features.columns:
    df_features["fin_flag_short_term_gearing_high"] = (
        df_features["fin_short_term_gearing"] > 100
    ).astype(int)

if "fin_debt_asset_ratio" in df_features.columns:
    df_features["fin_flag_debt_asset_high"] = (
        df_features["fin_debt_asset_ratio"] > 0.70
    ).astype(int)

    df_features["fin_flag_debt_asset_very_high"] = (
        df_features["fin_debt_asset_ratio"] > 0.85
    ).astype(int)

if "fin_long_term_liabilities_equity_ratio" in df_features.columns:
    df_features["fin_flag_long_term_liab_equity_high"] = (
        df_features["fin_long_term_liabilities_equity_ratio"] > 100
    ).astype(int)

if "fin_short_term_liabilities_equity_ratio" in df_features.columns:
    df_features["fin_flag_short_term_liab_equity_high"] = (
        df_features["fin_short_term_liabilities_equity_ratio"] > 100
    ).astype(int)

### Coverage / solvency features

Another important financial dimension is whether a supplier generates enough operating earnings to service debt, and whether its balance sheet shows adequate solvency.

The **interest coverage ratio** is commonly defined as:


$\text{Interest Coverage Ratio} = \frac{\text{EBIT}}{\text{Interest Expense}}$


This ratio measures how well operating earnings can cover interest payments. Low values indicate weaker debt-servicing capacity and are often interpreted as a sign of financial stress.  [oai_citation:3‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/commercial-lending/interest-coverage-ratio/?utm_source=chatgpt.com)

Solvency measures are broader indicators of longer-term financial resilience. Solvency ratios are commonly used to assess whether a company has enough assets and equity support to sustain its financial obligations over time.  [oai_citation:4‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/commercial-lending/solvency-ratio/?utm_source=chatgpt.com)

In this project, this group includes variables such as:

- `fin_interest_coverage_ratio`
- `fin_solvency_ratio_asset_based`

These are transformed into interpretable flags such as:

- `fin_flag_interest_coverage_stress`
- `fin_flag_interest_coverage_weak`
- `fin_flag_low_solvency`
- `fin_flag_very_low_solvency`

In [14]:
if "fin_interest_coverage_ratio" in df_features.columns:
    df_features["fin_flag_interest_coverage_stress"] = (
        df_features["fin_interest_coverage_ratio"] < 1
    ).astype(int)

    df_features["fin_flag_interest_coverage_weak"] = (
        df_features["fin_interest_coverage_ratio"] < 2
    ).astype(int)

if "fin_solvency_ratio_asset_based" in df_features.columns:
    df_features["fin_flag_low_solvency"] = (
        df_features["fin_solvency_ratio_asset_based"] < 20
    ).astype(int)

    df_features["fin_flag_very_low_solvency"] = (
        df_features["fin_solvency_ratio_asset_based"] < 10
    ).astype(int)

### Profitability stress features

Profitability measures reflect whether a supplier is generating earnings from its operations and revenues. Negative or persistently weak profitability can be a sign of financial distress and commercial vulnerability.

A standard profit margin is defined as:


$\text{Profit Margin} = \frac{\text{Net Income}}{\text{Revenue}}$


An operating margin such as EBIT margin is typically defined as:


$\text{EBIT Margin} = \frac{\text{EBIT}}{\text{Revenue}}$


These are standard profitability indicators used in financial analysis. Lower or negative values indicate weaker earnings performance.  [oai_citation:5‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/accounting/profit-margin/?utm_source=chatgpt.com)

In the EDA, `Profit_margin` and `EBIT_margin` were found to be highly correlated, so they represent similar information. For weak supervision, it is therefore often sufficient to engineer a simple profitability stress signal rather than rely on both raw variables separately.

The resulting engineered features include:

- `fin_flag_negative_profit_margin`
- `fin_flag_negative_ebit_margin`
- `fin_flag_profitability_stress`

In [15]:
if "fin_profit_margin" in df_features.columns:
    df_features["fin_flag_negative_profit_margin"] = (
        df_features["fin_profit_margin"] < 0
    ).astype(int)

if "fin_ebit_margin" in df_features.columns:
    df_features["fin_flag_negative_ebit_margin"] = (
        df_features["fin_ebit_margin"] < 0
    ).astype(int)

if {"fin_profit_margin", "fin_ebit_margin"}.issubset(df_features.columns):
    df_features["fin_flag_profitability_stress"] = (
        (df_features["fin_profit_margin"] < 0) |
        (df_features["fin_ebit_margin"] < 0)
    ).astype(int)

### Return stress features

Return-based measures describe how effectively a company turns assets or equity into profit.

Two standard ratios are:


$\text{ROA} = \frac{\text{Net Income}}{\text{Total Assets}}$



$\text{ROE} = \frac{\text{Net Income}}{\text{Shareholders' Equity}}$

ROA measures how effectively assets generate profit, while ROE measures returns delivered to shareholders. Both are standard indicators of financial performance and efficiency.  [oai_citation:6‡Corporate Finance Institute](https://corporatefinanceinstitute.com/resources/accounting/return-on-assets-roa-formula/?utm_source=chatgpt.com)

In this project, these are represented by:

- `fin_roa_net_income`
- `fin_roe_net_income`

These variables are mainly used as supporting stress indicators rather than the core financial labeling rules. They are converted into simple binary features such as:

- `fin_flag_negative_roa`
- `fin_flag_negative_roe`

In [16]:
if "fin_roe_net_income" in df_features.columns:
    df_features["fin_flag_negative_roe"] = (
        df_features["fin_roe_net_income"] < 0
    ).astype(int)

if "fin_roa_net_income" in df_features.columns:
    df_features["fin_flag_negative_roa"] = (
        df_features["fin_roa_net_income"] < 0
    ).astype(int)

### Composite financial stress features

In addition to individual financial indicators, the feature engineering step also creates composite stress measures that summarize the number of simultaneous warning signs observed for a supplier.

The idea is that financial risk is often not expressed through one variable alone, but through a combination of signals such as:

- weak rating
- low liquidity
- high leverage
- weak interest coverage
- low solvency
- negative profitability

To capture this, individual binary stress flags are summed into a total stress count:


$\text{Total Financial Stress Flags} = \sum_{j=1}^{J} \text{StressFlag}_j$


This produces a simple composite score that can then be thresholded into features such as:

- `fin_total_stress_flags`
- `fin_flag_multiple_financial_stress_signals`
- `fin_flag_severe_financial_stress`

In [17]:
stress_flag_cols = [
    "fin_flag_weak_implied_rating",
    "fin_flag_risk_take_caution_or_worse",
    "fin_flag_high_financial_risk_level",
    "fin_flag_financial_risk_score_high",
    "fin_flag_liquidity_stress",
    "fin_flag_short_term_gearing_high",
    "fin_flag_debt_asset_high",
    "fin_flag_interest_coverage_stress",
    "fin_flag_low_solvency",
    "fin_flag_profitability_stress",
    "fin_flag_negative_roe",
    "fin_flag_negative_roa",
]

existing_stress_flag_cols = [c for c in stress_flag_cols if c in df_features.columns]

if existing_stress_flag_cols:
    df_features["fin_total_stress_flags"] = df_features[existing_stress_flag_cols].sum(axis=1)

    df_features["fin_flag_multiple_financial_stress_signals"] = (
        df_features["fin_total_stress_flags"] >= 2
    ).astype(int)

    df_features["fin_flag_severe_financial_stress"] = (
        df_features["fin_total_stress_flags"] >= 3
    ).astype(int)

In [18]:
created_financial_features = [
    c for c in df_features.columns
    if c.startswith("fin_flag_")
    or c.endswith("_missing_flag")
    or c == "fin_implied_rating_ordinal"
    or c == "fin_total_stress_flags"
]

print(f"Number of created financial features: {len(created_financial_features)}")
print(created_financial_features)

display(df_features[created_financial_features].head())

Number of created financial features: 41
['fin_implied_rating_missing_flag', 'fin_risk_level_missing_flag', 'fin_financial_level_missing_flag', 'fin_financial_risk_score_missing_flag', 'fin_current_ratio_missing_flag', 'fin_gearing_missing_flag', 'fin_short_term_gearing_missing_flag', 'fin_long_term_gearing_missing_flag', 'fin_interest_coverage_ratio_missing_flag', 'fin_profit_margin_missing_flag', 'fin_ebit_margin_missing_flag', 'fin_implied_rating_ordinal', 'fin_flag_weak_implied_rating', 'fin_flag_moderate_or_worse_rating', 'fin_flag_risk_take_caution_or_worse', 'fin_flag_risk_do_not_source', 'fin_flag_high_financial_risk_level', 'fin_flag_financial_risk_score_high', 'fin_flag_financial_risk_score_very_high', 'fin_flag_liquidity_stress', 'fin_flag_severe_liquidity_stress', 'fin_flag_strong_liquidity', 'fin_flag_gearing_high', 'fin_flag_long_term_gearing_high', 'fin_flag_short_term_gearing_high', 'fin_flag_debt_asset_high', 'fin_flag_debt_asset_very_high', 'fin_flag_long_term_liab_eq

,fin_implied_rating_missing_flag,fin_risk_level_missing_flag,fin_financial_level_missing_flag,fin_financial_risk_score_missing_flag,fin_current_ratio_missing_flag,fin_gearing_missing_flag,fin_short_term_gearing_missing_flag,fin_long_term_gearing_missing_flag,fin_interest_coverage_ratio_missing_flag,fin_profit_margin_missing_flag,...,fin_flag_low_solvency,fin_flag_very_low_solvency,fin_flag_negative_profit_margin,fin_flag_negative_ebit_margin,fin_flag_profitability_stress,fin_flag_negative_roe,fin_flag_negative_roa,fin_total_stress_flags,fin_flag_multiple_financial_stress_signals,fin_flag_severe_financial_stress
0,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
2,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
3,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
4,1,1,1,1,1,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [19]:
print(df_final_no_spend.shape)
print(df_features.shape)

(9201, 147)
(9201, 188)


The financial view contains many raw accounting ratios and balance-sheet measures. For modeling, many of these raw variables can be retained. However, for weak supervision, simple and interpretable threshold-based indicators are more useful than raw continuous values.

Therefore, the financial feature engineering step transforms the raw financial variables into a smaller set of semantically meaningful stress indicators, covering:

- external/summarized risk
- liquidity stress
- leverage and liability stress
- debt-servicing stress
- solvency stress
- profitability stress
- return-based performance stress

## News Features

## ESG features

## Stock features

In [20]:
df_features = df_features.copy()

In [21]:
market_numeric_cols = [
    "vol_shock_ratio",
    "vol_trend_slope",
    "market_cap_volatility",
    "Price_trends_52 weeks_%",
    "market_beta_1y",
    "Earnings_per_share_DKK",
    "price_trend_slope"
]

for col in market_numeric_cols:
    if col in df_features.columns:
        df_features[col] = pd.to_numeric(df_features[col], errors="coerce")

Structural availability feature

In [22]:
stock_view_source_cols = [
    c for c in [
        "vol_shock_ratio",
        "market_cap_volatility",
        "Price_trends_52 weeks_%",
        "market_beta_1y",
        "Earnings_per_share_DKK",
        "price_trend_slope"
    ] if c in df_features.columns
]

if len(stock_view_source_cols) > 0:
    df_features["market_has_stock_view"] = (
        df_features[stock_view_source_cols].notna().any(axis=1)
    ).astype(int)

High volume shock Based on  EDA:  90% ~ 10.2, 95% ~ 19.5

In [23]:
if "vol_shock_ratio" in df_features.columns:
    df_features["market_flag_high_volume_shock"] = (
        df_features["vol_shock_ratio"] >= 10
    ).astype(int)


 High market cap volatility Based on EDA:  90% ~ 0.236, 95% ~ 0.299

In [24]:
if "market_cap_volatility" in df_features.columns:
    df_features["market_flag_high_market_cap_volatility"] = (
        df_features["market_cap_volatility"] >= 0.24
    ).astype(int)

Negative volume trend

In [25]:
if "vol_trend_slope" in df_features.columns:
    df_features["market_flag_negative_volume_trend"] = (
        df_features["vol_trend_slope"] < 0
    ).astype(int)

Negative price trend

In [26]:
if "price_trend_slope" in df_features.columns:
    df_features["market_flag_negative_price_trend"] = (
        df_features["price_trend_slope"] < 0
    ).astype(int)

Negative 52-week price trend

In [27]:
if "Price_trends_52 weeks_%" in df_features.columns:
    df_features["market_flag_negative_52w_price_trend"] = (
        df_features["Price_trends_52 weeks_%"] < 0
    ).astype(int)

High beta looked discrete in the EDA, often 0 / 1 / 2

In [28]:
if "market_beta_1y" in df_features.columns:
    df_features["market_flag_high_beta"] = (
        df_features["market_beta_1y"] >= 2
    ).astype(int)

 Negative EPS

In [29]:
if "Earnings_per_share_DKK" in df_features.columns:
    df_features["market_flag_negative_eps"] = (
        df_features["Earnings_per_share_DKK"] < 0
    ).astype(int)

Risk level from stock-price-based risk label

In [30]:
if "Risk level_stock_closing_price" in df_features.columns:
    df_features["market_flag_stock_price_take_caution_or_worse"] = (
        df_features["Risk level_stock_closing_price"].isin(["Take caution", "Do not source"])
    ).astype(int)

Log-transformed shock ratio since raw vol_shock_ratio is extremely skewed

In [31]:
if "vol_shock_ratio" in df_features.columns:
    df_features["market_log_vol_shock_ratio"] = np.log1p(df_features["vol_shock_ratio"])

Final kept market features

In [32]:
market_features_keep = [
    "market_has_stock_view",
    "market_flag_high_volume_shock",
    "market_flag_high_market_cap_volatility",
    "market_flag_negative_volume_trend",
    "market_flag_negative_price_trend",
    "market_flag_negative_52w_price_trend",
    "market_flag_high_beta",
    "market_flag_negative_eps",
    "market_flag_stock_price_take_caution_or_worse",
    "market_log_vol_shock_ratio",
    "market_total_stress_flags"
]

market_features_keep = [c for c in market_features_keep if c in df_features.columns]

print(market_features_keep)

display(df_features[market_features_keep].head())

['market_has_stock_view', 'market_flag_high_volume_shock', 'market_flag_high_market_cap_volatility', 'market_flag_negative_volume_trend', 'market_flag_negative_price_trend', 'market_flag_negative_52w_price_trend', 'market_flag_high_beta', 'market_flag_negative_eps', 'market_flag_stock_price_take_caution_or_worse', 'market_log_vol_shock_ratio']


,market_has_stock_view,market_flag_high_volume_shock,market_flag_high_market_cap_volatility,market_flag_negative_volume_trend,market_flag_negative_price_trend,market_flag_negative_52w_price_trend,market_flag_high_beta,market_flag_negative_eps,market_flag_stock_price_take_caution_or_worse,market_log_vol_shock_ratio
0,0,0,0,0,0,0,0,0,0,NaN
1,0,0,0,0,0,0,0,0,0,NaN
2,0,0,0,0,0,0,0,0,0,NaN
3,0,0,0,0,0,0,0,0,0,NaN
4,0,0,0,0,0,0,0,0,0,NaN


# Imputing 

In [33]:
df_audit = pd.DataFrame({
    "feature": df_features.columns,
    "dtype": df_features.dtypes.astype(str).values,
    "missing_pct": df_features.isna().mean().values * 100,
    "n_unique": df_features.nunique(dropna=True).values
})

def assign_view(col):
    if col.startswith("fin_"):
        return "financial"
    elif col.startswith("esg_"):
        return "esg"
    elif col.startswith("news_"):
        return "news"
    elif col in ["LPI_Score", "Customs", "Infrastructure", "International_Shipments",
                 "Logistics_Competence", "Tracking_Tracing", "Timeliness", "PPI_Value"]:
        return "macro_logistics"
    elif col in ["avg_vol", "std_vol", "max_vol", "min_vol", "vol_stability_score",
                 "vol_shock_ratio", "vol_trend_slope", "avg_market_cap",
                 "market_cap_volatility", "Price_trends_52 weeks_%", "market_beta_1y",
                 "Earnings_per_share_DKK", "Book_value_per_share_DKK", "avg_closing_price",
                 "price_volatility_score", "price_trend_slope"]:
        return "market"
    else:
        return "contract_core"

df_audit["view"] = df_audit["feature"].apply(assign_view)

display(
    df_audit.sort_values(["view", "missing_pct", "feature"])
)

,feature,dtype,missing_pct,n_unique,view
22,company_country,str,0.000000,15,contract_core
134,company_country_clean,str,0.000000,15,contract_core
30,contract_age_years,int64,0.000000,25,contract_core
19,contract_commodity,str,0.000000,139,contract_core
14,contract_currency_code,str,0.000000,16,contract_core
...,...,...,...,...,...
107,news_max_relevance_score,float64,87.120965,11,news
102,news_negative_count,float64,87.120965,8,news
108,news_negative_ratio,float64,87.120965,26,news
104,news_neutral_count,float64,87.120965,10,news


In [34]:
view_groups = {
    "financial": [c for c in df_features.columns if c.startswith("fin_") and not c.endswith("_flag") and not c.endswith("_missing_flag")],
    "esg": [c for c in df_features.columns if c.startswith("esg_")],
    "news": [c for c in df_features.columns if c.startswith("news_")],
}

for view_name, cols in view_groups.items():
    df_features[f"{view_name}_row_missing_pct"] = df_features[cols].isna().mean(axis=1) * 100

display(
    df_features[
        ["contract_id", "observation_year", "financial_row_missing_pct", "esg_row_missing_pct", "news_row_missing_pct"]
    ].head()
)

,contract_id,observation_year,financial_row_missing_pct,esg_row_missing_pct,news_row_missing_pct
0,9675,2018,64.634146,100.0,100.0
1,9675,2019,64.634146,100.0,100.0
2,9675,2020,64.634146,100.0,100.0
3,9675,2021,64.634146,100.0,100.0
4,9675,2022,64.634146,100.0,100.0


In [35]:
summary_by_view = (
    df_audit.groupby("view")
    .agg(
        n_features=("feature", "count"),
        avg_missing_pct=("missing_pct", "mean"),
        median_missing_pct=("missing_pct", "median"),
        high_missing_features=("missing_pct", lambda x: (x > 40).sum())
    )
    .reset_index()
)

display(summary_by_view)
display(
    df_audit.sort_values("missing_pct", ascending=False).head(50)
)

,view,n_features,avg_missing_pct,median_missing_pct,high_missing_features
0,contract_core,58,17.804661,0.000000,14
1,esg,14,89.001196,89.001196,14
2,financial,93,46.025853,76.089555,53
3,macro_logistics,8,0.767580,0.728182,0
4,market,16,98.322193,98.489295,16
5,news,9,87.120965,87.120965,9


,feature,dtype,missing_pct,n_unique,view
146,gold_department,str,99.087056,4,contract_core
145,gold_y,float64,99.087056,2,contract_core
197,market_log_vol_shock_ratio,float64,98.489295,48,contract_core
121,market_cap_volatility,float64,98.489295,48,market
112,avg_vol,float64,98.489295,48,market
113,std_vol,float64,98.489295,48,market
115,min_vol,float64,98.489295,48,market
116,vol_stability_score,float64,98.489295,48,market
117,vol_shock_ratio,float64,98.489295,48,market
118,vol_trend_slope,float64,98.489295,48,market


In [36]:
df_features.columns.to_list()

['contract_id',
 'contract_number',
 'contract_name',
 'contract_status',
 'contract_owner_cost_centre',
 'terminated',
 'term_type',
 'start_date',
 'expiration_date',
 'supplier_id',
 'supplier_number',
 'supplier_display_name',
 'payment_terms',
 'incoterms',
 'contract_currency_code',
 'contract_value',
 'contract_value_dkk',
 'contract_type',
 'nn_contract_type',
 'contract_commodity',
 'team',
 'unit',
 'company_country',
 'start_year',
 'expiry_year',
 'open_ended_contract',
 'end_year',
 'start_year_capped',
 'observation_year',
 'years_to_expiry',
 'contract_age_years',
 'expiry_pressure_bucket',
 'department_from_cost_center',
 'department',
 'moodys_bvd_id',
 'fin_moodys_company_name',
 'fin_closing_year',
 'fin_closing_date',
 'fin_created_at_utc',
 'fin_Status',
 'fin_implied_rating',
 'fin_risk_level',
 'fin_financial_level',
 'fin_output_text',
 'fin_Implied_rating - original',
 'fin_Number_of_months',
 'fin_Net_Salesth_DKK',
 'fin_Operating_revenue_Turnover_th_DKK',
 'f

# Save table

In [37]:
output_path = DATA_PROCESSED / "contracts_with_features.csv"

df_features.to_csv(output_path, index=False)

print("Saved feature matrix to:", output_path)
print("Shape:", df_features.shape)

Saved feature matrix to: /Users/Thomas/Desktop/Master Thesis/Data/processed/contracts_with_features.csv
Shape: (9201, 201)
